In [1]:
# (1) Mount drive, check for GPU, CUDA driver and pytorch versions
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi
import torch
print(torch.__version__)

Mounted at /content/drive
Tue Jan 10 21:26:34 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.32.03    Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  A100-SXM4-40GB      Off  | 00000000:00:04.0 Off |                    0 |
| N/A   33C    P0    46W / 400W |      0MiB / 40536MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                              

In [2]:
# (2) Clone PromCSE repo, install requirements and download dataset
%cd /content/drive/MyDrive/sent_embed_gce4/
!git clone https://github.com/YJiangcm/PromCSE.git

%cd PromCSE
!pip install -r requirements.txt
%cd SentEval/data/downstream/
!bash download_dataset.sh

%cd /content/drive/MyDrive/sent_embed_gce4/PromCSE/data/
!bash download_nli.sh
!bash download_wiki.sh
!pip install dill==0.3.4

## <Important> Replace PromCSE/promcse/models.py, PromCSE/promcse/trainers.py, and PromCSE/train.py

/content/drive/MyDrive/sent_embed_gce4
fatal: destination path 'PromCSE' already exists and is not an empty directory.
/content/drive/MyDrive/sent_embed_gce4/PromCSE
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 60.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 KB 22.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 91.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 70.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 97.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.7/790.7 KB 55.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 96.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.6/880.6 KB 65.0 MB/s eta 0:00:00
  Preparing 

In [6]:
# (4) Contrastive pretraining from bert-base, time required w/ 1 A100 GPU: [15 minutes/epoch]
%cd /content/drive/MyDrive/sent_embed_gce4/PromCSE

!python train.py \
    --model_name_or_path bert-base-uncased \
    --train_file data/nli_for_simcse.csv \
    --output_dir result/my-sup-promcse-eh-bert-base-2 \
    --num_train_epochs 10 \
    --per_device_train_batch_size 256 \
    --learning_rate 7e-3 \
    --max_seq_length 32 \
    --evaluation_strategy steps \
    --metric_for_best_model stsb_spearman \
    --load_best_model_at_end \
    --eval_steps 125 \
    --gcbs_quantile 0.999 \
    --pooler_type cls \
    --pre_seq_len 12 \
    --overwrite_output_dir \
    --temp 0.05 \
    --do_eh_loss \
    --eh_loss_margin 0.2 \
    --eh_loss_weight 10 \
    --do_train \
    --do_eval \
    --fp16 \
    "$@"

# Hyperparameters to tune (cmd line arguments)

# (1) Models: --model_name_or_path [bert-base-uncased, bert-large-uncased, roberta-base, roberta-large]
# (2) Learning rate: --learning_rate [range: (1e-3, 5e-2)]
# (3) Batch Size: --per_device_train_batch_size [64, 128, 256, 512]
# (4) GCBS Quantile: --gcbs_quantile [0.995, 0.999, 0.9995, 0.9999]
# (5) Number of Epochs: --num_train_epochs [3, 5, 7, 10, 15]
# (6) Prompt Sequence Length: --pre_seq_len [8, 10, 12]
# (7) Energy Loss (True/False): --do_eh_loss [remove to not use]
# (7) Energy Margin Size: --eh_loss_margin [0.15, 0.2, 0.25]
# (8) Energy Weight: eh_loss_weight [7, 10, 12]

/content/drive/MyDrive/sent_embed_gce4/PromCSE
2023-01-10 21:30:36.346337: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Using custom data configuration default
Reusing dataset csv (./data/csv/default-05c0c28d57c4f782/0.0.0/2960f95a26e85d40ca41a230ac88787f715ee3003edaacb8b1f0891e9f04dda2)
[INFO|configuration_utils.py:445] 2023-01-10 21:30:40,761 >> loading configuration file https://huggingface.co/bert-base-uncased/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/3c61d016573b14f7f008c02c4e51a366c67ab274726fe2910691e2a761acf43e.37395cee442ab11005bcd270f3c34464dc1704b715b5d7d52b1a461abe3b9e4e
[INFO|configuration_utils.py:481] 2023-01-10 21:30:40,762 >> Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_

In [7]:
# (5) Evaluate contrastive pretraining
%cd /content/drive/MyDrive/sent_embed_gce4/PromCSE
!python evaluation.py \
    --model_name_or_path result/my-sup-promcse-eh-bert-base-2 \
    --pooler_type cls \
    --task_set sts \
    --mode test \
    --pre_seq_len 12

/content/drive/MyDrive/sent_embed_gce4/PromCSE
total param is 109112832, trainable param is 221184
2023-01-11 00:01:34,903 : ***** Transfer task : STS12 *****


/content/drive/MyDrive/sent_embed_gce4/PromCSE/./SentEval/senteval/sts.py:46: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  sent1 = np.array([s.split() for s in sent1])[not_empty_idx]
/content/drive/MyDrive/sent_embed_gce4/PromCSE/./SentEval/senteval/sts.py:47: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  sent2 = np.array([s.split() for s in sent2])[not_empty_idx]
2023-01-11 00:01:37,561 : M